In [0]:
# from databricks.connect import DatabricksSession

# spark = DatabricksSession.builder.serverless().getOrCreate()


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType, LongType
from datetime import datetime, timedelta
import random

# Generate fake gamer behavior data
def generate_gamer_data(num_records=1000):
    data = []
    
    game_titles = ["Fortnite", "Call of Duty", "Minecraft", "League of Legends", "Valorant", "Apex Legends"]
    actions = ["login", "logout", "kill", "death", "level_up", "purchase", "achievement", "match_start", "match_end"]
    regions = ["NA", "EU", "ASIA", "SA", "OCE"]
    
    base_time = datetime.now() - timedelta(days=30)
    
    for i in range(num_records):
        player_id = f"player_{random.randint(1, 100)}"
        game_id = f"game_{random.choice(game_titles).replace(' ', '_').lower()}"
        session_id = f"session_{random.randint(1000, 9999)}"
        action = random.choice(actions)
        score = random.randint(0, 10000)
        playtime_minutes = random.randint(1, 240)
        level = random.randint(1, 100)
        in_game_currency = random.randint(0, 50000)
        real_money_spent = round(random.uniform(0, 100), 2)
        region = random.choice(regions)
        timestamp = base_time + timedelta(minutes=random.randint(0, 43200))
        event_date = timestamp.date()
        
        data.append((
            player_id,
            game_id,
            session_id,
            action,
            score,
            playtime_minutes,
            level,
            in_game_currency,
            real_money_spent,
            region,
            timestamp,
            str(event_date)  # Partition column in string format
        ))
    
    return data

# Create DataFrame with gamer behavior data
schema = StructType([
    StructField("player_id", StringType(), False),
    StructField("game_id", StringType(), False),
    StructField("session_id", StringType(), False),
    StructField("action", StringType(), False),
    StructField("score", IntegerType(), True),
    StructField("playtime_minutes", IntegerType(), True),
    StructField("level", IntegerType(), True),
    StructField("in_game_currency", IntegerType(), True),
    StructField("real_money_spent", DoubleType(), True),
    StructField("region", StringType(), True),
    StructField("event_timestamp", TimestampType(), False),
    StructField("event_date", StringType(), False)  # Partition column
])

# =========== Generate Gamer Data ==========

# gamer_data = generate_gamer_data(1000)
# df = spark.createDataFrame(gamer_data, schema)

# # Display sample data
# print("Generated Gamer Behavior Data:")
# print(f"\nTotal records: {df.count()}")
# print(f"\nSchema:")
# df.printSchema()

# # Write to Databricks Volume as Parquet
# # Note: Update the volume path to match your Databricks workspace
# volume_path = "/Volumes/jennifer_wang/gaming_demo/raw"

# try:
#     # Write partitioned by event_date
#     df.write \
#         .mode("overwrite") \
#         .partitionBy("event_date") \
#         .parquet(volume_path)
    
#     print(f"\n✓ Data successfully written to: {volume_path}")
    
#     # Verify the write
#     verify_df = spark.read.parquet(volume_path)
#     print(f"\n✓ Verified: Read back {verify_df.count()} records from volume")
    
# except Exception as e:
#     print(f"\n⚠ Error writing to volume: {e}")
#     print("\nNote: Make sure the volume path exists. You can create it using:")
#     print("CREATE VOLUME IF NOT EXISTS main.default.demo_volume;")


In [0]:
# Define paths
# source_path = "/Volumes/jennifer_wang/gaming_demo/raw"
source_path = "/Volumes/jennifer_wang/gaming_demo/raw/extended_batch" # uncomment this line to run the extended batch

checkpoint_path = "/Volumes/jennifer_wang/gaming_demo/checkpoints/gamer_events"
delta_table_path = "jennifer_wang.gaming_demo.gaming_events"


(spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "parquet")

  # Schema evolution options - automatically handle new columns
  .option("cloudFiles.schemaEvolutionMode", "addNewColumns")  # Add new columns automatically
  .option("cloudFiles.inferColumnTypes", "true")  # Infer column types
  .option("cloudFiles.schemaLocation", checkpoint_path)  # Store schema info
  
  # Additional Autoloader options
  .option("cloudFiles.useNotifications", "false")  # Use directory listing for demo
  .option("cloudFiles.includeExistingFiles", "true")  # Process existing files
  .load(source_path)
  
  .writeStream
  .option("checkpointLocation", checkpoint_path)
  .outputMode("append")
  .option("createTableColumnTypes", "")  # Ensures table schema is derived from data if creating
  .option("createTableIfNotExists", "true")  # Create Delta table if it does not exist
  .option("cloudFiles.schemaEvolutionMode", "addNewColumns")  # Add new columns automatically

  # Enable schema evolution on the Delta table side
  .option("checkpointLocation", checkpoint_path)
  .option("mergeSchema", "true")  # Allow schema evolution in Delta
  
  .trigger(availableNow=True)
  .toTable(delta_table_path))


In [0]:
# Verify the evolved schema
df = spark.sql(f"SELECT * FROM {delta_table_path}")

print(f"\n📊 Delta Table Statistics:")
print(f"   - Total records: {df.count()}")
print(f"   - Number of columns: {len(df.columns)}")

print("\n📋 Schema:")
df.printSchema()

print("\n📝 Sample Data:")
display(df)

In [0]:

# ============================================================================
# Demonstrate Schema Evolution - Simulate adding new columns
# ============================================================================
print("\n" + "=" * 70)
print("🔄 Demonstrating Schema Evolution")
print("=" * 70)

# Generate new data with additional columns (simulating schema evolution)
print("\n📝 Generating new data with additional columns...")

new_data_with_extra_cols = generate_gamer_data(100)  # Generate 100 more records

# Create DataFrame with NEW columns
extended_df = spark.createDataFrame(new_data_with_extra_cols, schema) \
    .withColumn("device_type", F.lit("mobile")) \
    .withColumn("connection_quality", F.lit("high")) \
    .withColumn("is_premium_user", F.lit(True))

print("✓ New columns added: device_type, connection_quality, is_premium_user")

# Write the extended data back to volume
extended_data_path = "/Volumes/jennifer_wang/gaming_demo/raw/extended_batch"
extended_df.write.mode("overwrite").parquet(extended_data_path)
print(f"✓ Extended data written to: {extended_data_path}")

print(f'Now change the new path to {extended_data_path} and run the previous cell to see the new schema. ')


In [0]:
# ============================================================================
# Cleaning up. 
# ============================================================================
dbutils.fs.rm("/Volumes/jennifer_wang/gaming_demo/checkpoints/gamer_events", True)
dbutils.fs.rm("/Volumes/jennifer_wang/gaming_demo/raw/extended_batch/", True)
spark.sql("DROP TABLE IF EXISTS jennifer_wang.gaming_demo.gaming_events")